In [2]:
# Libraries
import numpy as np
import pandas as pd
import deap

In [3]:
# Import prediction data
df = pd.read_csv("Results/RUL_predictions.csv")
RUL = dict(zip(df.engine_id, df.RUL_predicted))

In [4]:
# Various helper functions
def get_engine_due_date(engine_id, RUL):
    '''Computes the engine's safety due date (ts = t1 + Rj - 1), t1 = 1'''
    t1 = 1
    return t1 + RUL[engine_id] + 1

def get_mu_A(id):
    '''Returns the days to perform maintenance for team A'''
    if 1 <= id <= 20:
        return 5
    elif 21 <= id <= 55:
        return 3
    elif 56 <= id <= 80:
        return 4
    else:
        return 5
    
def get_maintenance_duration(engine_id, team_type):
    '''Returns the days to perform maintenance for the input team type and engine ID'''
    mu_A = get_mu_A(engine_id)

    if team_type == "A":
        return mu_A
    elif team_type == "B":
        if 1 <= engine_id <= 25:
            return mu_A - 1
        elif 26 <= engine_id <= 70:
            return mu_A + 3
        else:
            return mu_A + 2
    else:
        raise ValueError("get_maintenance_duration() received invalid team type!")
    
def get_cj_value(id):
    '''Returns the penalty value cj (c = cj * (t - ts)^2) for the engine id'''
    if 1 <= id <= 25:
        return 4
    elif 26 <= id <= 45:
        return 2
    elif 46 <= id <= 75:
        return 5
    else:
        return 6
    
def get_penalty_cost(engine_id, completion_date, due_date):
    '''Computes the penalty cost given the completion and due dates of an engine'''
    # In case completed in time, no penalty
    if completion_date <= due_date:
        return 0
    
    # Compute penalty value
    cj = get_cj_value(engine_id)
    delay = completion_date - due_date
    penalty = cj * (delay ** 2)
    return min(penalty, 250) # Return if less than 250, otherwise return 250



In [ ]:
# Encoding the GA individual (chromosomes)
# General idea: [(engine, team, start_day), (engine, team, start_day), ...]
# So one gene is one assignment of maintenance
# TODO